# 🎙️ NulTTS — OmniVoice TTS API

[![GitHub](https://img.shields.io/badge/GitHub-NulTTS-black?logo=github)](https://github.com/nultts/NulTTS) [![License](https://img.shields.io/badge/License-Apache--2.0-blue.svg)](https://github.com/nultts/NulTTS)

FastAPI Text-to-Speech API powered by OmniVoice.

**GitHub:** https://github.com/nultts/NulTTS

## ✨ Features

- 🎙️ OmniVoice TTS
- 🇻🇳 Vietnamese voice samples
- ⚡ CUDA acceleration
- 🌐 FastAPI REST API
- ☁️ Cloudflare Quick Tunnel
- 🛡️ Rate limit: 5 requests/minute
- 🎚️ Adjustable speech speed
- 🔊 WAV output at 24 kHz

**License:** Apache-2.0

## 📦 Install Dependencies

Cài đặt các thư viện cần thiết cho NulTTS.

In [ ]:
!pip install -q omnivoice fastapi uvicorn soundfile torch slowapi

## 🎤 Voice Samples

| Voice | Language |
|---|---|
| `Adam` | English |
| `minhanh` | Vietnamese |
| `namcongnghe` | Vietnamese |
| `namtramam` | Vietnamese |
| `ngochuyen` | Vietnamese |
| `nhongotngao` | Vietnamese |
| `thanhnientutin` | Vietnamese |

In [ ]:
import os
import gc
import re
import time
import subprocess
import threading
import torch
import soundfile as sf
import urllib.request

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

import uvicorn

from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded

from omnivoice import OmniVoice

In [ ]:
def freeport(port=8000):
    try:
        result = subprocess.check_output(f"lsof -t -i:{port}", shell=True).decode().split()
        for pid in result:
            subprocess.run(f"kill -9 {pid}", shell=True)
    except Exception:
        pass

freeport(8000)

limiter = Limiter(key_func=get_remote_address)

voices = {
    "Adam": "https://github.com/nultts/Voices/raw/main/adam.wav",
    "minhanh": "https://github.com/nultts/Voices/raw/main/minh_anh.wav",
    "namcongnghe": "https://github.com/nultts/Voices/raw/main/nam_cong_nghe.wav",
    "namtramam": "https://github.com/nultts/Voices/raw/main/nam_tram_am.wav",
    "ngochuyen": "https://github.com/nultts/Voices/raw/main/ngoc_huyen.wav",
    "nhongotngao": "https://github.com/nultts/Voices/raw/main/nho_ngot_ngao.wav",
    "thanhnientutin": "https://github.com/nultts/Voices/raw/main/thanh_nien_tu_tin.wav"
}

os.makedirs("voices", exist_ok=True)

for name, url in voices.items():
    file_path = f"voices/{name}.wav"
    if not os.path.exists(file_path):
        try:
            urllib.request.urlretrieve(url, file_path)
        except Exception:
            pass

## 🤖 Load OmniVoice

Model tự động sử dụng CUDA nếu GPU khả dụng và CPU nếu không có GPU.

In [ ]:
model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    load_asr=False,
)

## 🚀 FastAPI TTS Server

### Endpoint

`POST /api/tts`

### Request

```json
{
  "text": "Xin chào, đây là NulTTS!",
  "voice": "namcongnghe",
  "speed": 1.0
}
```

### Parameters

| Parameter | Type | Default | Description |
|---|---|---|---|
| `text` | string | — | Text cần chuyển thành giọng nói |
| `voice` | string | — | Voice sample |
| `speed` | float | `1.0` | Tốc độ đọc |

**Rate limit:** `5 requests/minute/IP`

In [ ]:
app = FastAPI(title="NulTTS API")

app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

class TTSRequest(BaseModel):
    text: str
    voice: str
    speed: float = 1.0

@app.post("/api/tts")
@limiter.limit("5/minute")
async def generate_tts(request: Request, req: TTSRequest):
    voice_path = f"voices/{req.voice}.wav"

    if not os.path.exists(voice_path):
        raise HTTPException(status_code=400, detail="Giọng này không tồn tại!")

    if not req.text.strip():
        raise HTTPException(status_code=400, detail="Text không được để trống!")

    if req.speed <= 0:
        raise HTTPException(status_code=400, detail="Speed phải lớn hơn 0!")

    try:
        output = model.generate(
            text=req.text,
            ref_audio=voice_path,
            num_step=32,
            guidance_scale=2.0,
            speed=req.speed,
            denoise=True,
            preprocess_prompt=True,
            postprocess_output=True,
            position_temperature=5.0,
            class_temperature=0.2,
            pad_duration=0.1,
            fade_duration=0.1
        )

        output_file = "output.wav"
        sf.write(output_file, output[0], 24000)

        return FileResponse(
            output_file,
            media_type="audio/wav",
            filename="output.wav"
        )
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 🌐 Start API & Cloudflare Tunnel

FastAPI chạy local trên `127.0.0.1:8000`.

Cloudflare Quick Tunnel sẽ tạo public URL dạng:

```text
https://xxxxx.trycloudflare.com
```

URL API sẽ được hiển thị sau khi tunnel khởi động.

In [ ]:
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

threading.Thread(target=start_server, daemon=True).start()

time.sleep(2)

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

def run_tunnel():
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    time.sleep(3)

    for line in proc.stderr:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)

        if match:
            url = match.group(0)
            print()
            print("=" * 70)
            print(f" NulTTS API : {url}")
            print(f" Endpoint   : {url}/api/tts")
            print(" GitHub     : https://github.com/nultts/NulTTS")
            print(" License    : Apache-2.0")
            print("=" * 70)
            print()
            return url

PUBLIC_URL = run_tunnel()

## 🧪 API Usage

### Python

```python
import requests

url = "YOUR_CLOUDFLARE_URL/api/tts"

data = {
    "text": "Xin chào! Đây là NulTTS.",
    "voice": "namcongnghe",
    "speed": 1.0
}

response = requests.post(url, json=data)

with open("output.wav", "wb") as f:
    f.write(response.content)
```

### JavaScript

```javascript
fetch("YOUR_CLOUDFLARE_URL/api/tts", {
  method: "POST",
  headers: {
    "Content-Type": "application/json"
  },
  body: JSON.stringify({
    text: "Xin chào! Đây là NulTTS.",
    voice: "namcongnghe",
    speed: 1.0
  })
})
.then(response => response.blob())
.then(blob => {
  const url = URL.createObjectURL(blob);
  const audio = new Audio(url);
  audio.play();
});
```

---

# 🎙️ NulTTS

Text-to-Speech API powered by **OmniVoice + FastAPI**.

🔗 **GitHub:** https://github.com/nultts/NulTTS

📜 **License:** Apache-2.0

⭐ If you find this project useful, consider starring the repository.

### Thanks for using NulTTS! ❤️

---